# Tarea 1 – Parte 1: Recolección y limpieza de noticias

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

| Configuración | Valor |
|---|---|
| Fecha de descarga | 18 de octubre de 2026 |
| Opción elegida | **B – El Comercio** (350–400 noticias, ~90–100 por categoría) |
| URL base | https://elcomercio.pe |
| Reglas del sitio | https://elcomercio.pe/robots.txt |
| Fuente principal (HTML) | `https://elcomercio.pe/archivo/<seccion>/` |
| Fuente de respaldo (RSS) | `https://elcomercio.pe/arcio/rss/category/<seccion>/` |

Categorías finales (fijas): `politica`, `economia`, `deportes`, `ciencia_tecnologia`.

**Plan:** 1) leer el `robots.txt`; 2) probar si el HTML responde o hay bloqueo;
3) si hay bloqueo en alguna sección, usar su RSS oficial y documentarlo.

## 1. Configuración

Buenas prácticas pedidas por el enunciado:
- `User-Agent` descriptivo (quiénes somos y para qué).
- Pausa de 1 a 2 segundos entre solicitudes.
- Máximo 1 reintento.
- Respaldo crudo de cada respuesta en `datos/raw/`.

In [1]:
import random
import re
import time
from datetime import date
from pathlib import Path
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests

BASE = "https://elcomercio.pe"
USER_AGENT = "QLAB-PythonIntermedio-Tarea1/1.0 (proyecto academico; titulares para clasificacion)"
PAUSA = (1.0, 2.0)          # segundos entre solicitudes
MAX_REINTENTOS = 1
RAW = Path("datos/raw")
RAW.mkdir(parents=True, exist_ok=True)

FECHA_DESCARGA = date.today().isoformat()
sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT})
print("Fecha de descarga:", FECHA_DESCARGA)

Fecha de descarga: 2026-10-18


## 2. Lectura del `robots.txt`

El `robots.txt` no es una herramienta de descarga: es el **reglamento** del sitio.
Lo descargamos, lo guardamos como evidencia y usamos `RobotFileParser` de la librería
estándar para preguntar, URL por URL, si nuestro `User-Agent` tiene permiso.

In [2]:
resp = sesion.get(f"{BASE}/robots.txt", timeout=20)
(RAW / "robots_elcomercio.txt").write_text(resp.text, encoding="utf-8")

# Reglas que aplican a todos los agentes (User-agent: *)
bloque_general = resp.text.split("# BLOQUEO")[0]
print(bloque_general.strip())

# DIRECTIVAS GENERALES
User-agent: *
Disallow: /catalogo/
Disallow: /buscar/
Disallow: /archivo/todas/19*
Disallow: /archivo/todas/200*


In [3]:
robots = RobotFileParser()
robots.parse(resp.text.splitlines())

# Una seccion del diario por cada categoria final
SECCIONES = {
    "politica": "politica",
    "economia": "economia",
    "deportes": "deporte-total",
    "ciencia_tecnologia": "tecnologia",
}

urls_a_revisar = []
for categoria, seccion in SECCIONES.items():
    urls_a_revisar += [
        (categoria, "HTML archivo", f"{BASE}/archivo/{seccion}/"),
        (categoria, "RSS", f"{BASE}/arcio/rss/category/{seccion}/"),
    ]
urls_a_revisar.append(("(control)", "ruta prohibida", f"{BASE}/buscar/economia/"))

permisos = pd.DataFrame(
    [(c, t, u, robots.can_fetch(USER_AGENT, u)) for c, t, u in urls_a_revisar],
    columns=["categoria", "tipo", "url", "permitido"],
)
permisos

,categoria,tipo,url,permitido
0,politica,HTML archivo,https://elcomercio.pe/archivo/politica/,True
1,politica,RSS,https://elcomercio.pe/arcio/rss/category/polit...,True
2,economia,HTML archivo,https://elcomercio.pe/archivo/economia/,True
3,economia,RSS,https://elcomercio.pe/arcio/rss/category/econo...,True
4,deportes,HTML archivo,https://elcomercio.pe/archivo/deporte-total/,True
5,deportes,RSS,https://elcomercio.pe/arcio/rss/category/depor...,True
6,ciencia_tecnologia,HTML archivo,https://elcomercio.pe/archivo/tecnologia/,True
7,ciencia_tecnologia,RSS,https://elcomercio.pe/arcio/rss/category/tecno...,True
8,(control),ruta prohibida,https://elcomercio.pe/buscar/economia/,False


La fila de control (`/buscar/`) debe salir `False`: así comprobamos que el parser
sí aplica las reglas y no responde `True` para todo.

## 3. Función de descarga responsable

`descargar()` hace una sola cosa con cuidado:
1. Si la URL ya está en `datos/raw/`, la lee del disco (no molesta al servidor otra vez).
2. Verifica el permiso en `robots.txt` antes de pedirla.
3. Hace la solicitud con pausa aleatoria de 1–2 s y **un** reintento si falla.
4. Guarda la respuesta cruda.

In [4]:
def nombre_cache(url: str) -> Path:
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", url.removeprefix(BASE)).strip("_") or "inicio"
    return RAW / f"{slug}.html"


def descargar(url: str) -> tuple[int, str]:
    """Devuelve (codigo_http, texto). Usa cache en datos/raw/ y respeta robots.txt."""
    archivo = nombre_cache(url)
    if archivo.exists():
        return 200, archivo.read_text(encoding="utf-8")
    if not robots.can_fetch(USER_AGENT, url):
        return -1, ""  # prohibido por robots.txt: ni siquiera se solicita

    codigo = 0  # 0 = sin respuesta (error de red)
    for intento in range(1 + MAX_REINTENTOS):
        time.sleep(random.uniform(*PAUSA))
        try:
            r = sesion.get(url, timeout=20)
        except requests.RequestException as e:
            print(f"  intento {intento + 1} fallo: {e}")
            continue
        codigo = r.status_code
        if codigo == 200:
            archivo.write_text(r.text, encoding="utf-8")
            return 200, r.text
        print(f"  intento {intento + 1}: HTTP {codigo}")
    return codigo, ""

## 4. Prueba de bloqueo

Un bloqueo puede verse de varias formas, no solo como un error:
- **HTTP 403 / 429 / 503**: el servidor rechaza o limita las solicitudes.
- **HTTP 200 pero sin noticias**: página de captcha, aviso de "acceso denegado"
  o contenido que se carga con JavaScript (que `requests` no ejecuta).

Por eso contamos cuántos titulares (`story-item__title`) trae realmente cada página HTML
y cuántos `<item>` trae cada RSS.

In [5]:
def contar_noticias(tipo: str, texto: str) -> int:
    if tipo == "RSS":
        return texto.count("<item>")
    return len(set(re.findall(r'class="story-item__title[^"]*"[^>]*href="([^"]+)"', texto)))


resultados = []
for categoria, tipo, url in urls_a_revisar[:-1]:   # sin la ruta de control
    codigo, texto = descargar(url)
    n = contar_noticias(tipo, texto) if codigo == 200 else 0
    if codigo == -1:
        estado = "prohibido por robots.txt"
    elif codigo != 200:
        estado = f"bloqueo / error HTTP {codigo}"
    elif n == 0:
        estado = "200 pero sin noticias (posible bloqueo)"
    else:
        estado = "OK"
    resultados.append((categoria, tipo, codigo, n, estado))

prueba = pd.DataFrame(resultados, columns=["categoria", "tipo", "http", "noticias", "estado"])
prueba

,categoria,tipo,http,noticias,estado
0,politica,HTML archivo,200,100,OK
1,politica,RSS,200,18,OK
2,economia,HTML archivo,200,100,OK
3,economia,RSS,200,16,OK
4,deportes,HTML archivo,200,100,OK
5,deportes,RSS,200,84,OK
6,ciencia_tecnologia,HTML archivo,200,100,OK
7,ciencia_tecnologia,RSS,200,13,OK


In [6]:
bloqueadas = prueba[(prueba.tipo == "HTML archivo") & (prueba.estado != "OK")]
if bloqueadas.empty:
    print("Sin bloqueo: todas las secciones responden por HTML. Fuente principal = HTML.")
    print("El RSS queda como respaldo automatico si una seccion fallara en la extraccion.")
else:
    print("Secciones con bloqueo en HTML (se usara su RSS):")
    print(bloqueadas[["categoria", "estado"]].to_string(index=False))

Sin bloqueo: todas las secciones responden por HTML. Fuente principal = HTML.
El RSS queda como respaldo automatico si una seccion fallara en la extraccion.
